In [0]:
from typing import Any

In [0]:
import logging
from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = "../../helpers/lng_lnd"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = "../../helpers/curated_helpers"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_deposit_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "DepositId string , SourceSystem string,"
        # "PortfolioKey string , PortfolioName string,"
        # "IngestionTimestamp timestamp , BatchId string,"
        # "DepositKey int , AccountNumber string, "
        # "PartyKey int , Currency string, "
        # "BankingUnitCode int , BankingUnit int, "
        # "LastBillingDate date , OnBalance decimal(15,2), "
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Deposit"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "profits",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== DepositAccount ======================
        deposit_account_extra_filters = None

        if deposit_account_extra_filters is not None:
            deposit_account_extra_filters.append(date_filter)
            deposit_account_final_filters = " AND ".join(deposit_account_extra_filters)
        else:
            deposit_account_final_filters = date_filter

        deposit_account_selected_fields = [
            "DEP_ACC_NUMBER",
            "DEP_OPEN_UNIT",
            "ACCOUNT_NUMBER",
            "CUST_ID",
            "CURRENCY",
            "DEP_OPEN_UNIT",
            "businessdate",
        ]

        deposit_account = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_depositaccount",
            deposit_account_final_filters,
            deposit_account_selected_fields,
        )

        # =================== DepositBalance   ====================

        deposit_balance_extra_filters = None

        if deposit_balance_extra_filters is not None:
            deposit_balance_extra_filters.append(date_filter)
            deposit_balance_final_filters = " AND ".join(deposit_balance_extra_filters)
        else:
            deposit_balance_final_filters = date_filter

        deposit_balance_selected_fields = [
            "LAST_TRX_DATE",
            "AVAILABLE_BALANCE",
            "DEP_ACC_NUMBER",
        ]

        deposit_balance = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_depositbalance",
            deposit_balance_final_filters,
            deposit_balance_selected_fields,
        )

        # Joins

        deposit_account_portfolio_join_condition = (
            deposit_account["DEP_OPEN_UNIT"] == portfolio["unit_code_profits"]
        )

        deposit_account_balance_join_condition = (
            deposit_account["dep_acc_number"] == deposit_balance["dep_acc_number"]
        )

        target_table = deposit_account.join(
            portfolio,
            deposit_account_portfolio_join_condition,
            "inner",
        ).join(
            deposit_balance,
            deposit_account_balance_join_condition,
            "left",
        )

        # HashKeys
        target_table_primary_key_fields = [
            deposit_account.DEP_ACC_NUMBER,
            lit("profits"),
        ]

        target_table = target_table.withColumn(
            "DepositId",
            sha2(
                concat_ws(";", *target_table_primary_key_fields),
                256,
            ),
        )

        #  ==================== Select of final columns ==========
        deposit_df = target_table.select(
            target_table.DepositId.alias("DepositId"),
            lit("profits").alias("SourceSystem"),
            deposit_account.DEP_OPEN_UNIT.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            deposit_account.DEP_ACC_NUMBER.alias("DepositKey"),
            deposit_account.ACCOUNT_NUMBER.alias("AccountNumber"),
            deposit_account.CUST_ID.alias("PartyKey"),
            deposit_account.CURRENCY.alias("Currency"),
            deposit_account.DEP_OPEN_UNIT.alias("BankingUnitCode"),
            deposit_account.DEP_OPEN_UNIT.alias("BankingUnit"),
            deposit_balance.LAST_TRX_DATE.alias("LastBillingDate"),
            deposit_balance.AVAILABLE_BALANCE.alias("OnBalance"),
            deposit_account.businessdate.alias("BusinessDate"),
        )

        # ================== Append =============================

        deposit_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_deposit_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)